<div style="
    background-color: #7BAFD4;
    border-radius: 18px;
    padding: 22px 26px;
    display: flex;
    align-items: center;
    gap: 24px;
    margin-bottom: 18px;
">
    <img src="592_avatar_lr.png" alt="GEOG 592"
        style="width:110px;height:110px;border-radius:22px;">
    <div>
        <div style="
            font-family: Impact, 'Arial Black', Arial, sans-serif;
            font-size: 42px;
            font-weight: 900;
            letter-spacing: 2px;
            line-height: 1;
            color: white;
        ">GEOG 592</div>
        <div style="
            font-family: Arial, Helvetica, sans-serif;
            font-size: 22px;
            font-weight: 700;
            color: #13294B;
            margin-top: 8px;
        ">GIS Programming</div>
        <div style="
            font-family: Arial, Helvetica, sans-serif;
            font-size: 16px;
            color: #13294B;
            margin-top: 6px;
        ">From Jupyter Notebook to a Reusable Python Script</div>
    </div>
</div>

#### Charlote Strom
#### 11/5/26

In [1]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

In [2]:
# Importing hog CAFO data
CAFOs = pd.read_csv(
    "/Users/Charlotte/Desktop/SeniorYear/GEOG_592/geog592-stromc/geog592-stromc/homework/Week07/data/List_Permitted_Animal_Facilities20260423.csv"
)

# Importing US Census county boundaries
url_0 = "https://www2.census.gov/geo/tiger/GENZ2024/shp/"
url_county = "cb_2024_us_county_500k.zip"

counties = gpd.read_file(url_0 + url_county)

In [3]:
# Selecting North Carolina counties
nc_counties = counties.loc[counties["STATEFP"] == "37"].to_crs(6543).copy()

nc_counties = nc_counties.sort_values("NAME").reset_index(drop=True)

In [4]:
# Selecting only hog CAFOs
Hog_CAFOs = CAFOs[
    CAFOs["Regulated Operation"] == "Swine"
].copy()

# Removing rows without coordinates
Hog_CAFOs = Hog_CAFOs.dropna(
    subset=["Location Lat Num", "Location Long Num"]
)

# Creating point geometry from longitude and latitude
Hog_CAFOs = gpd.GeoDataFrame(
    Hog_CAFOs,
    geometry=gpd.points_from_xy(
        Hog_CAFOs["Location Long Num"],
        Hog_CAFOs["Location Lat Num"]
    ),
    crs="EPSG:4326"
)

# Converting CAFO points to the same CRS as counties
Hog_CAFOs = Hog_CAFOs.to_crs(6543)


In [5]:
# Spatial join: assigning each CAFO to its county
joined = gpd.sjoin(
    Hog_CAFOs,
    nc_counties[["NAME", "geometry"]],
    how="left",
    predicate="within"
)

# Counting CAFOs per county
cafo_counts = joined.groupby("NAME").size()

# Including counties with zero CAFOs
cafo_counts = cafo_counts.reindex(
    nc_counties["NAME"],
    fill_value=0
)

# Total number of hog CAFOs in NC
total_cafos = cafo_counts.sum()

print("Total hog CAFOs:", total_cafos)

Total hog CAFOs: 2117


In [6]:
def county_map(county, ax):

    selected = nc_counties[nc_counties["NAME"] == county]

    # Plot county boundary
    selected.plot(
        ax=ax,
        edgecolor="black",
        facecolor="#7b9b90"
    )

    # Select CAFOs located in this county
    county_cafos = joined[joined["NAME"] == county]

    # Plot CAFO points
    county_cafos.plot(
        ax=ax,
        color="red",
        markersize=12,
        marker="o"
    )

    ax.set_axis_off()


In [7]:
def locator_map(county, ax):

    selected = nc_counties[nc_counties["NAME"] == county]

    # Plot all NC counties
    nc_counties.plot(
        ax=ax,
        edgecolor="gray",
        facecolor="white"
    )

    # Highlight selected county
    selected.plot(
        ax=ax,
        edgecolor="black",
        facecolor="#7b9b90"
    )

    ax.set_axis_off()

In [8]:
def county_pie(county, ax):

    cafo_count = cafo_counts[county]

    other_cafos = total_cafos - cafo_count

    ax.pie(
        [cafo_count, other_cafos],
        labels=["This County", "Rest of NC"],
        colors=["red", "lightgray"],
        autopct="%.1f%%",
        startangle=90
    )

    ax.set_title("Share of NC Hog CAFOs")

In [9]:
def county_info(county, ax):

    cafo_count = cafo_counts[county]

    # CAFO icon / marker
    ax.scatter(
        0.15, 0.70,
        color="red",
        s=250,
        transform=ax.transAxes
    )

    ax.text(
        0.35, 0.75,
        "HOG CAFOs",
        transform=ax.transAxes,
        fontsize=14,
        color="red",
        va="center"
    )

    ax.text(
        0.35, 0.45,
        f"{cafo_count} permitted facilities",
        transform=ax.transAxes,
        fontsize=14,
        va="center"
    )

    ax.text(
        0.15, 0.15,
        f"North Carolina total: {total_cafos}",
        transform=ax.transAxes,
        fontsize=12,
        color="gray"
    )

    ax.set_axis_off()


In [10]:
def county_figure(county, pdf):

    fig = plt.figure(figsize=(12, 8))

    # Creating axes for each atlas component
    ax_county = fig.add_axes([0.05, 0.23, 0.55, 0.54])
    ax_locator = fig.add_axes([0.05, 0.08, 0.25, 0.12])
    ax_pie = fig.add_axes([0.65, 0.48, 0.30, 0.30])
    ax_info = fig.add_axes([0.65, 0.12, 0.30, 0.32])

    # Calling the functions
    county_map(county, ax_county)
    locator_map(county, ax_locator)
    county_pie(county, ax_pie)
    county_info(county, ax_info)

    # Page heading
    fig.text(
        0.05, 0.94,
        "NORTH CAROLINA / Hog CAFO Atlas",
        fontsize=10,
        color="olivedrab"
    )

    # County title
    fig.text(
        0.05, 0.87,
        f"{county} County",
        fontsize=28,
        fontweight="bold"
    )

    # Footer
    fig.text(
        0.05, 0.03,
        "Source: NC DEQ Permitted Animal Facilities | "
        "U.S. Census Bureau TIGER/Line 2024",
        fontsize=9,
        color="gray"
    )

    # Saving atlas page
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


In [11]:
with PdfPages("NC_CAFO_Atlas.pdf") as pdf:

    for county in nc_counties["NAME"]:
        county_figure(county, pdf)


/var/folders/k4/0d_fkbsj5gs7vl1cm28c39kr0000gn/T/ipykernel_64518/2920712299.py:16: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  county_cafos.plot(
/var/folders/k4/0d_fkbsj5gs7vl1cm28c39kr0000gn/T/ipykernel_64518/2920712299.py:16: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  county_cafos.plot(
/var/folders/k4/0d_fkbsj5gs7vl1cm28c39kr0000gn/T/ipykernel_64518/2920712299.py:16: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  county_cafos.plot(
/var/folders/k4/0d_fkbsj5gs7vl1cm28c39kr0000gn/T/ipykernel_64518/2920712299.py:16: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  county_cafos.plot(
/var/folders/k4/0d_fkbsj5gs7vl1cm28c39kr0000gn/T/ipykernel_64518/2920712299.py:16: UserWarning: The GeoDataFrame you are attempting to plot is empty. Nothing has been displayed.
  county_cafos.plot(
/var/